# Transform playground

Notatnik do szybkiego testowania funkcji z `transform/transform.py` przeciwko już zapisanym
danym Bronze (`data/raw/`), bez odpalania całego `pipeline.py`.

`%autoreload 2` przeładowuje moduły projektu przed każdą komórką, więc zmiany w `transform/transform.py` są widoczne bez restartu kernela.

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import pandas as pd
import config

## BRONZE

In [6]:
df = pd.read_json(RAW_DIR / RAW_NOAA_EARTHQUAKES_FILENAME)
df.shape

(6698, 49)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6698 entries, 0 to 6697
Data columns (total 49 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   id                               6698 non-null   int64  
 1   year                             6698 non-null   int64  
 2   locationName                     6698 non-null   str    
 3   latitude                         6635 non-null   float64
 4   longitude                        6635 non-null   float64
 5   eqMagnitude                      4912 non-null   float64
 6   damageAmountOrder                4908 non-null   float64
 7   eqMagUnk                         840 non-null    float64
 8   publish                          6698 non-null   bool   
 9   country                          6698 non-null   str    
 10  regionCode                       6698 non-null   int64  
 11  intensity                        3443 non-null   float64
 12  deathsAmountOrder              

## SILVER

# Transform earthquakes

In [8]:
from transform.transform import transform_noaa_earthquakes

result = transform_noaa_earthquakes()
result.info()

<class 'pandas.DataFrame'>
RangeIndex: 6698 entries, 0 to 6697
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                6698 non-null   int64  
 1   year              6698 non-null   int64  
 2   month             6294 non-null   Int64  
 3   day               6123 non-null   Int64  
 4   country           6698 non-null   str    
 5   latitude          6635 non-null   float64
 6   longitude         6635 non-null   float64
 7   magnitude         4912 non-null   float64
 8   deaths_total      6698 non-null   int64  
 9   volcano_event_id  82 non-null     Int64  
dtypes: Int64(3), float64(3), int64(3), str(1)
memory usage: 589.0 KB


In [9]:
result.dtypes

id                    int64
year                  int64
month                 Int64
day                   Int64
country                 str
latitude            float64
longitude           float64
magnitude           float64
deaths_total          int64
volcano_event_id      Int64
dtype: object

# Read clean

Earthquake

In [11]:
earthquake_df = pd.read_parquet(config.CLEAN_DIR / config.CLEAN_NOAA_EARTHQUAKES_FILENAME)
earthquake_df.shape

(6698, 10)

Volcanos

In [12]:
volcano_df = pd.read_parquet(config.CLEAN_DIR / config.CLEAN_NOAA_VOLCANOES_FILENAME)
volcano_df.shape

(903, 14)

In [34]:
v_df = pd.read_parquet(config.CLEAN_DIR/config.CLEAN_NOAA_VOLCANOES_FILENAME)

cols = ["volcano_location_id", "name", "country", "latitude", "longitude", "elevation", "morphology"]

dim_volcano = v_df[cols]
dim_volcano.groupby("volcano_location_id").nunique().max()



name          1
country       1
latitude      1
longitude     1
elevation     1
morphology    1
dtype: int64

In [36]:
dim_volcano = dim_volcano.drop_duplicates(subset="volcano_location_id").reset_index(drop=True)
dim_volcano.head

<bound method NDFrame.head of      volcano_location_id           name           country  latitude  \
0                  10106           Etna             Italy    37.748   
1                  30301       Karthala           Comoros   -11.750   
2                  50107  Ritter Island  Papua New Guinea    -5.519   
3                  50214         Rabaul  Papua New Guinea    -4.246   
4                  20303     Nyiragongo        Congo, DRC    -1.520   
..                   ...            ...               ...       ...   
274               170102      Reykjanes           Iceland    63.817   
275               140210           Agua         Guatemala    14.465   
276               201091    Hayli Gubbi          Ethiopia    13.510   
277               804091        Sofugan             Japan    29.794   
278              1502824          Altar           Ecuador    -1.670   

     longitude  elevation          morphology  
0       14.999       3357       Stratovolcano  
1       43.380       

In [39]:
len(dim_volcano)


279

In [40]:
dim_volcano["volcano_location_id"].is_unique

True

In [41]:
dim_volcano.isna().sum()

volcano_location_id    0
name                   0
country                0
latitude               0
longitude              0
elevation              0
morphology             0
dtype: int64

In [48]:
from transform.schemas import dim_volcano_schema

dim_volcano_schema.validate(dim_volcano, lazy=True)

,volcano_location_id,name,country,latitude,longitude,elevation,morphology
0,10106,Etna,Italy,37.748,14.999,3357,Stratovolcano
1,30301,Karthala,Comoros,-11.750,43.380,2361,Shield volcano
2,50107,Ritter Island,Papua New Guinea,-5.519,148.115,75,Stratovolcano
3,50214,Rabaul,Papua New Guinea,-4.246,152.194,688,Pyroclastic shield
4,20303,Nyiragongo,"Congo, DRC",-1.520,29.250,3470,Stratovolcano
...,...,...,...,...,...,...,...
274,170102,Reykjanes,Iceland,63.817,-22.717,140,Crater rows
275,140210,Agua,Guatemala,14.465,-90.743,3760,Stratovolcano
276,201091,Hayli Gubbi,Ethiopia,13.510,40.722,493,Shield volcano
277,804091,Sofugan,Japan,29.794,140.342,99,Submarine volcano
